# Tema 3. Modelos lineales generalizados con regularización: el código de las slides

Este cuaderno reproduce, en el orden de las slides, las figuras y cifras del deck del tema 3 (familias de GLM, elección del modelo y GLM con regularización). Usa los mismos datos, semillas y modelos, así que los resultados coinciden con los de las slides.

- **Cómo usarlo**: ejecuta las celdas en orden (*Restart & Run All*). Cada sección indica a qué slide corresponde.
- **Entorno**: el del curso (`tap`): scikit-learn, pandas, numpy y matplotlib.
- **Tiempo**: menos de un minuto (las secciones de las slides 15, 16, 18 y 19 ajustan muchos modelos).
- **Datos**: anuncios de Airbnb en Valencia (Inside Airbnb), los mismos del tema 2. Se descargan de la web del curso.

El código prioriza la lectura: un paso por línea, nombres descriptivos y un comentario por paso.

**Solo si trabajas en Google Colab.** La celda siguiente instala las mismas versiones de los paquetes que el entorno `tap` del curso. En tu ordenador, con el entorno `tap` activado, no hace nada. Si Colab pide reiniciar la sesión después de instalar, acepta y vuelve a ejecutar el cuaderno desde el principio.

In [ ]:
# Instala las versiones del curso (solo en Google Colab)
import sys
if "google.colab" in sys.modules:
    %pip install -q -r https://raw.githubusercontent.com/juanDmontoro/tap/main/requirements.txt

## 0. Configuración

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.inspection import partial_dependence
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import (LinearRegression, LogisticRegression, LogisticRegressionCV,
                                  GammaRegressor, PoissonRegressor)
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.model_selection import (train_test_split, KFold, StratifiedKFold, cross_val_predict,
                                     cross_val_score, cross_validate, GridSearchCV)
from sklearn.metrics import log_loss, root_mean_squared_error

# Colores del curso
AZUL, MARINO, TERRACOTA, SALVIA, GRIS = "#447099", "#1F3A5F", "#E07A5F", "#81B29A", "#6C757D"
plt.rcParams.update({"figure.figsize": (8, 5), "axes.spines.top": False, "axes.spines.right": False})

# Un barrio que no aparece al ajustar se codifica como fila de ceros; scikit-learn lo avisa en cada
# predicción. Es lo esperado, así que silenciamos ese aviso.
warnings.filterwarnings("ignore", message="Found unknown categories")

SEED = 42

## 1. Datos

Misma población y misma división que el tema 2: anuncios entre 30 y 400 € por noche, alojamientos enteros y habitaciones privadas; un 30 % de los anuncios se reserva como conjunto de test. En este cuaderno solo se usa el conjunto de entrenamiento.

In [ ]:
URL_DATOS = "https://juandmontoro.github.io/tap/data/listados.csv"
columnas = ["neighbourhood_cleansed", "room_type", "latitude", "longitude", "accommodates",
            "bathrooms", "bedrooms", "beds", "number_of_reviews", "number_of_reviews_ltm",
            "reviews_per_month", "review_scores_rating", "minimum_nights", "availability_365",
            "host_is_superhost", "price"]
df = pd.read_csv(URL_DATOS, usecols=columnas)

# El precio viene como texto ("$1,200.00"): quitamos "$" y "," y lo pasamos a número
df["price"] = pd.to_numeric(df["price"].str.replace(r"[$,]", "", regex=True))

# Distancia al centro (plaza del Ayuntamiento) en km
lat0, lon0 = 39.4699, -0.3763
dif_norte = (df["latitude"] - lat0) * 111.0
dif_este = (df["longitude"] - lon0) * 111.0 * np.cos(np.radians(lat0))
df["dist_km"] = np.sqrt(dif_norte ** 2 + dif_este ** 2)

# Superanfitrión como 0/1 ("t" = sí)
df["superhost"] = (df["host_is_superhost"] == "t").astype(int)

# Población: precio entre 30 y 400 €, alojamientos enteros y habitaciones privadas
en_rango = df["price"].between(30, 400)
tipo_valido = df["room_type"].isin(["Entire home/apt", "Private room"])
datos = df[en_rango & tipo_valido].copy()
datos["log_precio"] = np.log(datos["price"])

# División entrenamiento/test (la misma de los temas 1 y 2)
entreno, test = train_test_split(datos, test_size=0.3, random_state=SEED)
print("Anuncios de entrenamiento:", len(entreno))

### Preprocesado común

El mismo `Pipeline` del tema 2 para todos los modelos: las variables numéricas se imputan con la mediana y se estandarizan; el barrio y el tipo de alojamiento se codifican con una categoría de referencia (el barrio más frecuente y la habitación privada). Los barrios con menos de 50 anuncios se agrupan en una categoría común.

In [ ]:
num = ["dist_km", "accommodates", "bathrooms", "bedrooms", "beds", "number_of_reviews",
       "number_of_reviews_ltm", "reviews_per_month", "review_scores_rating", "minimum_nights",
       "availability_365", "superhost"]
cat = ["neighbourhood_cleansed", "room_type"]

barrio_referencia = datos["neighbourhood_cleansed"].value_counts().idxmax()
REF = [barrio_referencia, "Private room"]


def preprocesador(numericas, min_barrio=50):
    """Imputación y estandarización de las numéricas; barrio y tipo con categoría de referencia.

    Los barrios con menos de `min_barrio` anuncios se agrupan en una categoría común.
    Con min_barrio=None, cada barrio tiene su propia columna (slides 16 y 18).
    """
    pasos_num = Pipeline([("imp", SimpleImputer(strategy="median")),
                          ("esc", StandardScaler())])
    pasos_cat = OneHotEncoder(drop=REF, min_frequency=min_barrio, handle_unknown="infrequent_if_exist",
                              sparse_output=False)
    return ColumnTransformer([("num", pasos_num, numericas),
                              ("cat", pasos_cat, cat)])


# Validación cruzada en 5 particiones (estratificada para las respuestas de clase)
cv = KFold(5, shuffle=True, random_state=SEED)
cv_estrat = StratifiedKFold(5, shuffle=True, random_state=SEED)

### RMSE de validación cruzada

En las slides, el RMSE de validación es la media del RMSE de las 5 particiones. `cross_val_score` lo devuelve con signo negativo (scikit-learn maximiza las puntuaciones), así que le cambiamos el signo.

In [ ]:
def rmse_cv(modelo, X, y):
    """RMSE de cada una de las 5 particiones de validación cruzada."""
    puntuaciones = cross_val_score(modelo, X, y, cv=cv, scoring="neg_root_mean_squared_error")
    return -puntuaciones

## Slide 2. ¿Sirve la regresión lineal para estas cuatro variables?

Cuatro respuestas sobre los mismos anuncios: precio (positivo), superanfitrión (sí/no), tramo de precio (tres clases) y reseñas del último año (recuento).

In [ ]:
# Tramos de precio: cortes en los terciles del precio de entrenamiento
cortes = entreno["price"].quantile([1 / 3, 2 / 3]).to_numpy()
NOMBRES_TRAMO = ["bajo", "medio", "alto"]
tramo = pd.cut(entreno["price"], [0, cortes[0], cortes[1], np.inf], labels=NOMBRES_TRAMO).astype(str)
print("Cortes de los tramos (€):", cortes)

# Proporción de superanfitriones (sin los anuncios sin dato, como en el tema 2)
ent_c = entreno[entreno["host_is_superhost"].notna()]
prevalencia = ent_c["superhost"].mean()

fig, axes = plt.subplots(2, 2, figsize=(10, 7))

axes[0, 0].hist(entreno["price"], bins=40, color=AZUL)
axes[0, 0].set_title("precio (€/noche)")

axes[0, 1].bar(["no", "sí"], [1 - prevalencia, prevalencia], color=[GRIS, AZUL], width=0.6)
axes[0, 1].set_title("¿superanfitrión?")

reparto_tramo = tramo.value_counts(normalize=True)[NOMBRES_TRAMO]
axes[1, 0].bar(NOMBRES_TRAMO, reparto_tramo, color=[SALVIA, AZUL, TERRACOTA], width=0.6)
axes[1, 0].set_title("tramo de precio")

# Reseñas truncadas en 100 para que se vea la forma
axes[1, 1].hist(entreno["number_of_reviews_ltm"].clip(upper=100), bins=50, color=AZUL)
axes[1, 1].set_title("reseñas del último año")

fig.tight_layout()
plt.show()

ceros = (entreno["number_of_reviews_ltm"] == 0).mean()
print(f"Superanfitriones: {prevalencia:.1%}. Anuncios sin reseñas en el último año: {ceros:.1%}")

## Slide 5. Una recta da probabilidades fuera de [0, 1]

Modelo de probabilidad lineal (regresión lineal con respuesta 0/1) y logit, con los predictores del tema 2: los de la semana 2 del tema 1 sin superanfitrión y con log(precio). Los dos se ajustan con todo el conjunto de entrenamiento.

In [ ]:
# Predictores del modelo de superanfitrión
num_c = ["dist_km", "accommodates", "bathrooms", "bedrooms", "beds", "number_of_reviews",
         "number_of_reviews_ltm", "reviews_per_month", "review_scores_rating", "minimum_nights",
         "availability_365", "log_precio"]
X_c = ent_c[num_c + cat]
y_c = ent_c["superhost"]

# Modelo de probabilidad lineal
mpl = Pipeline([("prep", preprocesador(num_c)), ("model", LinearRegression())])
mpl.fit(X_c, y_c)
pred_mpl = mpl.predict(X_c)

# Logit sin penalización (C = infinito), el del tema 2
logit = Pipeline([("prep", preprocesador(num_c)),
                  ("model", LogisticRegression(C=np.inf, max_iter=5000))])
logit.fit(X_c, y_c)
prob_logit = logit.predict_proba(X_c)[:, 1]          # columna 1 = probabilidad de superanfitrión

# ¿Cuántas predicciones del modelo lineal salen de [0, 1]?
fuera = (pred_mpl < 0) | (pred_mpl > 1)
print(f"Fuera de [0, 1]: {fuera.mean():.1%} (por debajo de 0: {(pred_mpl < 0).mean():.1%}, "
      f"por encima de 1: {(pred_mpl > 1).mean():.1%})")

# Predicciones de los dos modelos, con los anuncios ordenados de menor a mayor predicción lineal
orden = np.argsort(pred_mpl)
percentil = 100 * np.arange(len(orden)) / (len(orden) - 1)

fig, ax = plt.subplots()
ax.scatter(percentil, prob_logit[orden], s=8, color=AZUL, alpha=0.35, label="logit")
ax.scatter(percentil, pred_mpl[orden], s=8, color=TERRACOTA, alpha=0.6, label="regresión lineal")
ax.axhline(0, color=GRIS, ls="--")
ax.axhline(1, color=GRIS, ls="--")
ax.set_xlabel("anuncios ordenados por la predicción de la regresión lineal (percentil)")
ax.set_ylabel("probabilidad predicha")
ax.set_ylim(-1, 1.4)                     # eje recortado: un valor extremo aplasta la escala
print("Predicciones fuera del eje:", ((pred_mpl < -1) | (pred_mpl > 1.4)).sum())
ax.legend()
plt.show()

## Slide 7. Los coeficientes se estiman por máxima verosimilitud

Con distribución de Bernoulli, la log-verosimilitud de cada anuncio es $\log \mu_i$ si es superanfitrión y $\log(1-\mu_i)$ si no lo es. Dividida por el número de anuncios, es la log-loss del tema 2 con el signo cambiado: maximizar una equivale a minimizar la otra. Lo comprobamos con el logit ajustado en la slide 5. En la sección de la slide 10 se repite con el multinomial.

In [ ]:
n = len(y_c)
log_verosimilitud = np.sum(y_c * np.log(prob_logit) + (1 - y_c) * np.log(1 - prob_logit))

print("Log-verosimilitud / n:", round(log_verosimilitud / n, 4))
print("Log-loss del modelo:  ", round(log_loss(y_c, prob_logit), 4))

## Slide 9. Superanfitrión: distribución de Bernoulli y enlace logit

`decision_function` devuelve el predictor lineal $\nu_i$ de cada anuncio; la probabilidad es $1/(1+e^{-\nu_i})$.

In [ ]:
# Predictor lineal de cada anuncio
nu = logit.decision_function(X_c)

# Comprobación: aplicar la inversa del enlace da la misma probabilidad que predict_proba
prob_desde_nu = 1 / (1 + np.exp(-nu))
print("Diferencia máxima con predict_proba:", np.abs(prob_desde_nu - prob_logit).max())

# Anuncios en 0 o 1 con un poco de ruido vertical para que se vean
rng = np.random.default_rng(0)
ruido = rng.uniform(-0.03, 0.03, len(y_c))
rejilla_nu = np.linspace(-8, 6, 300)

fig, ax = plt.subplots()
ax.scatter(nu, y_c + ruido, s=6, color=GRIS, alpha=0.25)
ax.plot(rejilla_nu, 1 / (1 + np.exp(-rejilla_nu)), color=AZUL, lw=3)
ax.set_xlim(-8, 6)
ax.set_xlabel("predictor lineal ν")
ax.set_ylabel("P(superanfitrión)")
plt.show()

## Slide 10. Tramo de precio: el logit multinomial

`LogisticRegression` con una respuesta de tres clases ajusta el logit multinomial. El precio no entra como predictor (define el tramo).

La figura asigna a todos los anuncios la misma capacidad, calcula sus probabilidades y las promedia, para capacidades de 1 a 8. Es la **dependencia parcial**, y scikit-learn la calcula con `partial_dependence` (exige que la columna sea de tipo `float`).

In [ ]:
X_t = entreno[num + cat]
multi = Pipeline([("prep", preprocesador(num)),
                  ("model", LogisticRegression(C=np.inf, max_iter=5000))])
multi.fit(X_t, tramo)
print("Orden de las clases en predict_proba:", multi.classes_)

# Dependencia parcial: probabilidad media de cada tramo con todos los anuncios en la misma capacidad
CAPACIDADES = list(range(1, 9))
X_t_float = X_t.astype({"accommodates": float})        # partial_dependence exige float
dependencia = partial_dependence(multi, X_t_float, ["accommodates"], response_method="predict_proba",
                                 kind="average", custom_values={"accommodates": CAPACIDADES})

# "average" tiene una fila por tramo y una columna por capacidad: la trasponemos
prob_por_capacidad = pd.DataFrame(dependencia["average"].T, index=CAPACIDADES, columns=multi.classes_)
print(prob_por_capacidad[NOMBRES_TRAMO].round(2))

fig, ax = plt.subplots()
ax.stackplot(CAPACIDADES,
             prob_por_capacidad["bajo"], prob_por_capacidad["medio"], prob_por_capacidad["alto"],
             colors=[SALVIA, AZUL, TERRACOTA], labels=NOMBRES_TRAMO)
ax.set_xlabel("capacidad (plazas)")
ax.set_ylabel("probabilidad media del tramo")
ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
plt.show()

Comprobación de la slide 7 con tres clases: la log-verosimilitud de cada anuncio es el logaritmo de la probabilidad predicha de su tramo observado. Dividida por el número de anuncios, es la log-loss multiclase con el signo cambiado.

In [ ]:
probabilidades = multi.predict_proba(X_t)                  # columnas en el orden de multi.classes_
columna_observada = np.searchsorted(multi.classes_, tramo)  # columna del tramo de cada anuncio
prob_observada = probabilidades[np.arange(len(tramo)), columna_observada]

print("Log-verosimilitud / n:", round(np.log(prob_observada).mean(), 4))
print("Log-loss multiclase:  ", round(log_loss(tramo, probabilidades), 4))

## Slide 11. Precio: distribución Gamma y enlace logarítmico

GLM Gamma con enlace log (`GammaRegressor`, sin penalización: `alpha=0`) y todos los predictores de la semana 2 del tema 1. Predicciones de validación cruzada: cada anuncio se predice con un modelo estimado sin él.

In [ ]:
X_p = entreno[num + cat]
y_p = entreno["price"]

gamma = Pipeline([("prep", preprocesador(num)),
                  ("model", GammaRegressor(alpha=0, max_iter=5000))])
pred_gamma = cross_val_predict(gamma, X_p, y_p, cv=cv)

# Desviación típica del precio observado en cada décimo de anuncios según el precio predicho
decil = pd.qcut(pred_gamma, 10, labels=False)
resumen_deciles = pd.DataFrame({"precio": y_p.to_numpy(), "decil": decil})
resumen_deciles = resumen_deciles.groupby("decil")["precio"].agg(["mean", "std"])
print(resumen_deciles.round(1))

fig, ax = plt.subplots()
ax.scatter(pred_gamma, y_p, s=6, color=AZUL, alpha=0.25)
ax.plot([30, 330], [30, 330], color=TERRACOTA, lw=2)       # diagonal: predicción = observación
ax.set_xlabel("precio medio predicho (€)")
ax.set_ylabel("precio observado (€)")
plt.show()

## Slide 12. Reseñas del último año: distribución de Poisson

`PoissonRegressor` con enlace log. Predictores: los del tema 1 con log(precio), sin las otras variables de reseñas (contienen la misma información) ni la valoración media (falta cuando no hay reseñas).

La figura agrupa los anuncios en diez grupos del mismo tamaño según las reseñas predichas por validación cruzada y compara, en cada grupo, la media predicha con la media observada. scikit-learn no tiene una función para esta figura: los grupos se forman con `pd.qcut`.

In [ ]:
num_r = ["dist_km", "accommodates", "bathrooms", "bedrooms", "beds", "minimum_nights",
         "availability_365", "superhost", "log_precio"]
X_r = entreno[num_r + cat]
y_r = entreno["number_of_reviews_ltm"]

poisson = Pipeline([("prep", preprocesador(num_r)),
                    ("model", PoissonRegressor(alpha=0, max_iter=5000))])
pred_pois = cross_val_predict(poisson, X_r, y_r, cv=cv)

# Diez grupos según las reseñas predichas: media predicha y media observada en cada uno
grupo = pd.qcut(pred_pois, 10, labels=False)
por_grupo = pd.DataFrame({"predicha": pred_pois, "observada": y_r.to_numpy(), "grupo": grupo})
por_grupo = por_grupo.groupby("grupo")[["predicha", "observada"]].mean()
print(por_grupo.round(1))

fig, ax = plt.subplots()
limite = por_grupo.max().max() * 1.1
ax.plot([0, limite], [0, limite], color=TERRACOTA, lw=2)      # diagonal: media predicha = observada
ax.scatter(por_grupo["predicha"], por_grupo["observada"], s=80, color=AZUL)
ax.set_xlabel("media predicha de reseñas en el grupo")
ax.set_ylabel("media observada de reseñas en el grupo")
plt.show()

In [ ]:
# Regresión lineal con los mismos predictores: ¿cuántos recuentos negativos predice?
lineal_r = Pipeline([("prep", preprocesador(num_r)), ("model", LinearRegression())])
pred_lineal_r = cross_val_predict(lineal_r, X_r, y_r, cv=cv)
print(f"Predicciones negativas de la regresión lineal: {(pred_lineal_r < 0).mean():.1%}")
print(f"Predicción más baja del Poisson: {pred_pois.min():.2g}")

# RMSE de validación (notas de la slide)
print("RMSE Poisson:          ", round(rmse_cv(poisson, X_r, y_r).mean(), 1))
print("RMSE regresión lineal: ", round(rmse_cv(lineal_r, X_r, y_r).mean(), 1))
print("RMSE nulo:             ", round(rmse_cv(DummyRegressor(strategy="mean"), X_r, y_r).mean(), 1))

El error de las predicciones del Poisson en reseñas, con el RMSE y el MAE, frente al modelo nulo.

In [ ]:
# RMSE y MAE medios de las 5 particiones: Poisson y modelo nulo
for nombre, modelo in [("Nulo (media)", DummyRegressor(strategy="mean")), ("Poisson", poisson)]:
    resultado = cross_validate(modelo, X_r, y_r, cv=cv,
                               scoring=["neg_root_mean_squared_error", "neg_mean_absolute_error"])
    rmse_medio = -resultado["test_neg_root_mean_squared_error"].mean()
    mae_medio = -resultado["test_neg_mean_absolute_error"].mean()
    print(f"{nombre}: RMSE {rmse_medio:.1f} reseñas, MAE {mae_medio:.1f} reseñas")

## Slide 15. ¿Qué modelo elegimos para el precio?

Tres candidatos para el precio, evaluados en las mismas 5 particiones. La métrica principal es el RMSE, que pesa más los errores grandes. El MAE (error absoluto medio) y el R² se muestran como alternativas:

- regresión lineal sobre el precio;
- MCO sobre log(precio), como en el tema 1, con la predicción devuelta a euros con la exponencial;
- GLM Gamma con enlace log.

MCO sobre log(precio) se ajusta con el logaritmo y se evalúa en euros. `TransformedTargetRegressor` aplica el logaritmo a la respuesta antes de ajustar y la exponencial a la predicción, así que `cross_val_score` da el RMSE en euros.

**Aviso**: la celda ajusta 20 modelos y tarda unos segundos.

In [ ]:
lineal_p = Pipeline([("prep", preprocesador(num)), ("model", LinearRegression())])
# MCO sobre log(precio): ajusta con log(precio) y predice exp(predicción), en euros
mco_log = TransformedTargetRegressor(regressor=Pipeline([("prep", preprocesador(num)),
                                                         ("model", LinearRegression())]),
                                     func=np.log, inverse_func=np.exp)

# RMSE de cada partición para los cuatro modelos
rmse_nulo = rmse_cv(DummyRegressor(strategy="mean"), X_p, y_p)
rmse_lineal = rmse_cv(lineal_p, X_p, y_p)
rmse_mco = rmse_cv(mco_log, X_p, y_p)
rmse_gamma = rmse_cv(gamma, X_p, y_p)

# Predicciones de validación cruzada de MCO sobre log(precio), en euros
pred_mco = cross_val_predict(mco_log, X_p, y_p, cv=cv)

tabla_rmse = pd.DataFrame({"RMSE de validación (€)": [rmse_nulo.mean(), rmse_lineal.mean(),
                                                       rmse_mco.mean(), rmse_gamma.mean()]},
                          index=["Nulo (precio medio)", "Regresión lineal", "MCO sobre log(precio)", "GLM Gamma"])
print(tabla_rmse.round(1))

In [ ]:
# Diferencias por partición entre candidatos: media y error típico (orientativo, tema 2)
for nombre, diferencia in [("lineal - MCO log", rmse_lineal - rmse_mco),
                           ("lineal - Gamma", rmse_lineal - rmse_gamma),
                           ("MCO log - Gamma", rmse_mco - rmse_gamma)]:
    error_tipico = diferencia.std(ddof=1) / np.sqrt(len(diferencia))
    print(f"{nombre}: {diferencia.mean():.2f} ± {error_tipico:.2f} €")

# Rango de las predicciones de validación cruzada
pred_lineal_p = cross_val_predict(lineal_p, X_p, y_p, cv=cv)
print(f"Precios negativos de la regresión lineal: {(pred_lineal_p < 0).mean():.2%}")
print(f"Precio predicho más bajo: Gamma {pred_gamma.min():.1f} €, MCO sobre log {pred_mco.min():.1f} €")

# ¿Qué predice cada modelo? Media de las predicciones frente al precio medio
print(f"Precio medio observado:          {y_p.mean():.1f} €")
print(f"Media de las predicciones Gamma: {pred_gamma.mean():.1f} €")
print(f"Media de las predicciones MCO:   {pred_mco.mean():.1f} €")

Las mismas comparaciones con el MAE y el R². `cross_validate` acepta una lista de métricas y devuelve una puntuación por partición para cada una (con signo negativo en las que son errores).

**Aviso**: la celda ajusta 20 modelos y tarda unos segundos.

In [ ]:
# Los cuatro modelos del precio
modelos_precio = {"Nulo (precio medio)": DummyRegressor(strategy="mean"),
                  "Regresión lineal": lineal_p,
                  "MCO sobre log(precio)": mco_log,
                  "GLM Gamma": gamma}

# MAE y R² medios de las 5 particiones para cada modelo
filas = []
for nombre, modelo in modelos_precio.items():
    resultado = cross_validate(modelo, X_p, y_p, cv=cv, scoring=["neg_mean_absolute_error", "r2"])
    mae_medio = -resultado["test_neg_mean_absolute_error"].mean()
    r2_medio = resultado["test_r2"].mean()
    filas.append({"modelo": nombre, "MAE (€)": mae_medio, "R²": r2_medio})

tabla_mae_r2 = pd.DataFrame(filas).set_index("modelo")
tabla_precio = tabla_rmse.join(tabla_mae_r2)
print(tabla_precio.round(2))

Con el MAE queda por delante MCO sobre log(precio): devuelve la mediana del precio, y la mediana es la predicción que minimiza el error absoluto. El Gamma estima la media.

## Slide 16. La métrica de clasificación depende del problema

Superanfitrión: el 30 % de los anuncios lo es, así que las clases están desequilibradas. Se calculan cuatro métricas para el modelo nulo y el logit con `cross_validate`, en las mismas particiones estratificadas:

- log-loss (`neg_log_loss`): calidad de las probabilidades;
- AUC (`roc_auc`) y precisión media, AP (`average_precision`): si el modelo ordena bien a los superanfitriones;
- exactitud (`accuracy`): proporción de aciertos con umbral 0,5.

El modelo nulo predice la proporción de cada clase.

In [ ]:
# Métricas de clasificación (nombre en la tabla: nombre en scikit-learn)
metricas = {"log-loss": "neg_log_loss", "AUC": "roc_auc", "AP": "average_precision", "exactitud": "accuracy"}

nulo_clases = DummyClassifier(strategy="prior")
modelos_clase = {"Nulo (proporción)": nulo_clases, "Logit": logit}

# Media de las 5 particiones de cada métrica para cada modelo
filas = []
for nombre, modelo in modelos_clase.items():
    resultado = cross_validate(modelo, X_c, y_c, cv=cv_estrat, scoring=metricas)
    fila = {"modelo": nombre}
    for metrica in metricas:
        fila[metrica] = resultado[f"test_{metrica}"].mean()
    filas.append(fila)

tabla_superanfitrion = pd.DataFrame(filas).set_index("modelo")
tabla_superanfitrion["log-loss"] = -tabla_superanfitrion["log-loss"]    # la log-loss se puntúa en negativo
print(tabla_superanfitrion.round(3))

Tramo de precio: tres clases del mismo tamaño (los terciles del precio) y ningún error más costoso que otro, así que la exactitud es una métrica razonable. El AUC y la AP se definen para dos clases.

In [ ]:
# Proporción de anuncios en cada tramo
print(tramo.value_counts(normalize=True).round(3))

logloss_multi = -cross_val_score(multi, X_t, tramo, cv=cv_estrat, scoring="neg_log_loss").mean()
logloss_nulo_t = -cross_val_score(nulo_clases, X_t, tramo, cv=cv_estrat, scoring="neg_log_loss").mean()
exactitud_multi = cross_val_score(multi, X_t, tramo, cv=cv_estrat, scoring="accuracy").mean()
exactitud_nulo = cross_val_score(nulo_clases, X_t, tramo, cv=cv_estrat, scoring="accuracy").mean()
print(f"Tramo, log-loss: nulo {logloss_nulo_t:.2f}, multinomial {logloss_multi:.2f}")
print(f"Tramo, exactitud: nulo {exactitud_nulo:.0%}, multinomial {exactitud_multi:.0%}")

## Slide 17. Con pocos anuncios, el Gamma sobreajusta

El mismo GLM Gamma del precio, sin penalización, estimado con dos conjuntos:

- 300 anuncios del entrenamiento (los de la semana 2 del tema 1: misma semilla), con una columna para cada barrio;
- los 5 324 anuncios del entrenamiento, con el preprocesado de las demás slides.

`cross_validate(..., return_train_score=True)` da, para cada partición, el RMSE con los anuncios que estiman el modelo (entrenamiento) y con la partición que queda fuera (validación).

In [ ]:
# 300 anuncios del entrenamiento, con una columna para cada barrio
entreno_300 = entreno.sample(300, random_state=SEED)
X_300 = entreno_300[num + cat]
y_300 = entreno_300["price"]
gamma_300 = Pipeline([("prep", preprocesador(num, min_barrio=None)),
                      ("model", GammaRegressor(alpha=0, max_iter=5000))])
print("Barrios distintos entre los 300:", entreno_300["neighbourhood_cleansed"].nunique())

# RMSE de entrenamiento y de validación, media de las 5 particiones
ajuste_300 = cross_validate(gamma_300, X_300, y_300, cv=cv, scoring="neg_root_mean_squared_error",
                            return_train_score=True)
ajuste_todo = cross_validate(gamma, X_p, y_p, cv=cv, scoring="neg_root_mean_squared_error",
                             return_train_score=True)
rmse_ent_300 = -ajuste_300["train_score"].mean()
rmse_val_300 = -ajuste_300["test_score"].mean()
rmse_ent_todo = -ajuste_todo["train_score"].mean()
rmse_val_todo = -ajuste_todo["test_score"].mean()
rmse_nulo_300 = rmse_cv(DummyRegressor(strategy="mean"), X_300, y_300).mean()

print(f"300 anuncios: entrenamiento {rmse_ent_300:.1f} €, validación {rmse_val_300:.1f} €, nulo {rmse_nulo_300:.1f} €")
print(f"{len(entreno)} anuncios: entrenamiento {rmse_ent_todo:.1f} €, validación {rmse_val_todo:.1f} €")

fig, ax = plt.subplots()
posiciones = np.arange(2)
ax.bar(posiciones - 0.2, [rmse_ent_300, rmse_ent_todo], width=0.38, color=AZUL, label="entrenamiento")
ax.bar(posiciones + 0.2, [rmse_val_300, rmse_val_todo], width=0.38, color=TERRACOTA, label="validación")
ax.hlines(rmse_nulo_300, -0.45, 0.45, color=GRIS, ls="--")    # modelo nulo con 300 anuncios
ax.set_xticks(posiciones, ["300 anuncios", f"{len(entreno)} anuncios"])
ax.set_ylabel("RMSE (€)")
ax.legend()
plt.show()

In [ ]:
# Anuncio con el mayor precio predicho en validación (300 anuncios, sin penalización)
pred_300 = cross_val_predict(gamma_300, X_300, y_300, cv=cv)
peor = np.argmax(pred_300)
print(entreno_300.iloc[peor][["neighbourhood_cleansed", "room_type", "price"]].to_dict(),
      "| precio predicho:", round(pred_300[peor]))

## Slide 18. Solución: verosimilitud penalizada

Un GLM con penalización maximiza $\ell(\beta) - \lambda \cdot \text{penalización}(\beta)$. En scikit-learn:

- `GammaRegressor` y `PoissonRegressor`: λ se llama `alpha` y la penalización es la de Ridge (no admiten Lasso).
- `LogisticRegression`: `C` = 1/λ (a mayor `C`, menos penalización) y `l1_ratio` fija la mezcla: 0 = Ridge, 1 = Lasso. Con Lasso hace falta `solver="saga"` (o `"liblinear"`).

La penalización va siempre dentro del `Pipeline`, después de estandarizar los predictores.

## Slide 19. ¿Cuándo necesita penalización un GLM?

`GridSearchCV` prueba cada valor de `alpha` con validación cruzada en 5 particiones y puntúa con el RMSE (`scoring="neg_root_mean_squared_error"`). El eje logarítmico no admite `alpha = 0`: el modelo sin penalización de la slide 17 se dibuja como una línea horizontal.

**Aviso**: la celda ajusta 110 modelos y tarda unos segundos.

In [ ]:
# Valores de la penalización alpha (de casi nula a fuerte)
alphas = np.logspace(-4, 1, 11)

# Curva de validación con 300 anuncios
busqueda_300 = GridSearchCV(gamma_300, {"model__alpha": alphas}, cv=cv,
                            scoring="neg_root_mean_squared_error")
busqueda_300.fit(X_300, y_300)
rmse_curva_300 = -busqueda_300.cv_results_["mean_test_score"]

# Curva de validación con todo el entrenamiento (el modelo `gamma` de la slide 11)
busqueda_todo = GridSearchCV(gamma, {"model__alpha": alphas}, cv=cv,
                             scoring="neg_root_mean_squared_error")
busqueda_todo.fit(X_p, y_p)
rmse_curva_todo = -busqueda_todo.cv_results_["mean_test_score"]

print(pd.DataFrame({"alpha": alphas.round(4), "RMSE con 300": rmse_curva_300,
                    "RMSE con todo": rmse_curva_todo}).round({"RMSE con 300": 2, "RMSE con todo": 2}))
print("alpha elegido con 300 anuncios:", busqueda_300.best_params_["model__alpha"],
      "| RMSE:", round(-busqueda_300.best_score_, 1))
print("alpha elegido con todo el entrenamiento:", round(busqueda_todo.best_params_["model__alpha"], 4))

fig, ax = plt.subplots()
ax.plot(alphas, rmse_curva_todo, color=AZUL, marker="o", label=f"{len(entreno)} anuncios")
ax.plot(alphas, rmse_curva_300, color=TERRACOTA, marker="o", label="300 anuncios")
ax.axhline(rmse_val_300, color=TERRACOTA, ls="--", lw=1)        # 300 anuncios sin penalización
ax.set_xscale("log")
ax.set_xlabel("penalización alpha (escala log)")
ax.set_ylabel("RMSE de validación cruzada (€)")
ax.legend()
plt.show()

Ya con `alpha = 0.0001` el RMSE baja mucho: la penalización modera el precio predicho más alto. Comprobación fuera de la validación cruzada: los anuncios del entrenamiento que no están entre los 300 (el test sigue reservado).

In [ ]:
# Precio predicho más alto, sin penalización y con la penalización más pequeña de la rejilla
gamma_300_min = clone(gamma_300).set_params(model__alpha=alphas[0])
pred_300_min = cross_val_predict(gamma_300_min, X_300, y_300, cv=cv)
print(f"Precio predicho más alto: {pred_300.max():.0f} € con alpha = 0 y {pred_300_min.max():.0f} € con alpha = {alphas[0]}")

# Anuncios de entrenamiento no usados en el ajuste de los 300
resto = entreno.drop(entreno_300.index)
alpha_elegido = busqueda_300.best_params_["model__alpha"]
for alpha in [0, alpha_elegido]:
    modelo = clone(gamma_300).set_params(model__alpha=alpha)
    modelo.fit(X_300, y_300)
    rmse_resto = root_mean_squared_error(resto["price"], modelo.predict(resto[num + cat]))
    print(f"alpha = {alpha}: RMSE en los {len(resto)} anuncios no usados = {rmse_resto:.1f} €")

## Slide 20. Selección de predictores con Lasso (en el modelo logit)

`LogisticRegressionCV` busca `C` por validación cruzada. Con `l1_ratios=[1.0]` la penalización es la de Lasso, y `scoring="neg_log_loss"` puntúa con la log-loss. Se queda con el `C` de menor log-loss (`C_`).

La regla del error típico (tema 2) elige el `C` más pequeño (más penalización) cuya log-loss queda a menos de un error típico del mínimo. Se calcula con `scores_`, que guarda la puntuación de cada partición y cada `C`.

**Aviso**: la celda ajusta unos 55 modelos y tarda unos segundos.

In [ ]:
valores_C = np.logspace(-3, 1, 9)

logit_lasso_cv = Pipeline([("prep", preprocesador(num_c)),
                           ("model", LogisticRegressionCV(Cs=valores_C, l1_ratios=[1.0], solver="saga",
                                                          cv=cv_estrat, scoring="neg_log_loss",
                                                          max_iter=20000, use_legacy_attributes=False))])
logit_lasso_cv.fit(X_c, y_c)
print("C elegido por LogisticRegressionCV:", logit_lasso_cv["model"].C_)

# scores_ tiene forma (particiones, l1_ratios, valores de C); la puntuación es -log-loss
logloss_particiones = -logit_lasso_cv["model"].scores_[:, 0, :]
logloss_media = logloss_particiones.mean(axis=0)
logloss_ee = logloss_particiones.std(axis=0, ddof=1) / np.sqrt(5)

# Coeficientes distintos de cero para cada C (modelo ajustado con todo el entrenamiento)
activos = []
for valor in valores_C:
    lasso = Pipeline([("prep", preprocesador(num_c)),
                      ("model", LogisticRegression(C=valor, l1_ratio=1.0, solver="saga", max_iter=20000))])
    lasso.fit(X_c, y_c)
    activos.append(np.count_nonzero(lasso["model"].coef_))

# Regla del error típico: el C más pequeño con log-loss <= mínimo + su error típico
k_min = np.argmin(logloss_media)
umbral = logloss_media[k_min] + logloss_ee[k_min]
k_1ee = np.min(np.where(logloss_media <= umbral)[0])
print("Mínimo: C =", valores_C[k_min], "con", activos[k_min], "coeficientes")
print("Regla del error típico: C =", round(valores_C[k_1ee], 4), "con", activos[k_1ee], "coeficientes")
print(f"Diferencia de log-loss: {1000 * (logloss_media[k_1ee] - logloss_media[k_min]):.1f} milésimas")

print(pd.DataFrame({"C": valores_C, "log-loss": logloss_media, "error típico": logloss_ee,
                    "coeficientes activos": activos}).round(4))

fig, ax = plt.subplots()
ax.errorbar(valores_C, logloss_media, yerr=logloss_ee, color=AZUL, marker="o", capsize=4)
ax.axvline(valores_C[k_min], color=MARINO, ls="--")      # mínimo
ax.axvline(valores_C[k_1ee], color=TERRACOTA, ls="--")   # regla del error típico
for valor, perdida, error, n_activos in zip(valores_C, logloss_media, logloss_ee, activos):
    ax.annotate(str(n_activos), (valor, perdida + error), textcoords="offset points",
                xytext=(0, 6), ha="center")
ax.set_xscale("log")
ax.set_xlabel("C = 1/λ (escala log)")
ax.set_ylabel("log-loss de validación cruzada")
plt.show()

## Slide 21. ¿Qué dice un coeficiente igual a cero?

Coeficientes del logit con el `C` de la regla del error típico, y correlación entre la capacidad y los dormitorios.

In [ ]:
lasso_1ee = Pipeline([("prep", preprocesador(num_c)),
                      ("model", LogisticRegression(C=valores_C[k_1ee], l1_ratio=1.0, solver="saga",
                                                   max_iter=20000))])
lasso_1ee.fit(X_c, y_c)

coeficientes = pd.Series(lasso_1ee["model"].coef_[0], index=lasso_1ee["prep"].get_feature_names_out())
print("Coeficientes distintos de cero:")
print(coeficientes[coeficientes != 0].round(3))
print()
print("Coeficientes en cero:", (coeficientes == 0).sum(), "de", len(coeficientes))
print("Correlación capacidad-dormitorios:", round(X_c["accommodates"].corr(X_c["bedrooms"]), 2))